# Finding 1: everything correlates with everything

We overlooked this in A3, but also were mosly focused on a subset of variables to create interpretable DAGs with. But there are many more variables in the set that correlate with stress (or each other) that would be helpful to put into a model.

Many are likely caused by stress, rather than the other way around. All the stuff like blood pessure.

In [2]:
import pandas as pd

df = pd.read_csv("../data/StressLevelDataset.csv")

In [3]:
treatments = ["study_load", "living_conditions", "noise_level",
              "academic_performance", "future_career_concerns"]
others = [c for c in df.columns if c not in treatments + ["stress_level"]]

# how strongly each variable you left out relates to each treatment and to stress
corr = df.corr(method="spearman")
print(corr.loc[others, treatments + ["stress_level"]].round(2))

                              study_load  living_conditions  noise_level  \
anxiety_level                       0.63              -0.62         0.64   
self_esteem                        -0.62               0.61        -0.62   
mental_health_history               0.56              -0.54         0.55   
depression                          0.64              -0.59         0.60   
headache                            0.59              -0.59         0.59   
blood_pressure                      0.38              -0.34         0.39   
sleep_quality                      -0.60               0.59        -0.62   
breathing_problem                   0.47              -0.50         0.49   
safety                             -0.54               0.60        -0.58   
basic_needs                        -0.56               0.56        -0.62   
teacher_student_relationship       -0.57               0.60        -0.58   
social_support                     -0.51               0.52        -0.53   
peer_pressur

## Finding 2: The first is 0.63, basically contributes a lot and our R^2 is expected to be high, very high. This can be a problem for DML because what does the ML even have to learn at this point? Also confounding issues with T and might find low CATE. Like if stress is 63% caused by ___, the control vs treatment won't make a huge difference

In [2]:
import numpy as np
eig = np.linalg.eigvalsh(df.corr(method="spearman"))[::-1]
print((eig / eig.sum())[:3].round(2))  # if the first is ~0.6+, it's essentially one factor

[0.63 0.05 0.03]


# Trial 1: simple linear regression for the outcome variable, XGBoost for the treatment model. Simple linear regression for the residual-on-residual bit.

In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

Y = df["stress_level"].values

# confounders from the data check (likely stress symptoms excluded)
confounders = ["mental_health_history", "safety", "basic_needs", "social_support",
               "teacher_student_relationship", "peer_pressure", "bullying",
               "extracurricular_activities"]

# DAG4 structure among treatments, plus the new confounders
adj = {
    "study_load": confounders,
    "living_conditions": confounders,
    "noise_level": confounders + ["living_conditions"],
    "academic_performance": confounders + ["living_conditions", "study_load"],
    "future_career_concerns": confounders + ["academic_performance"],
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def dml(Y, T, W):
    # stage 1: predict Y from W (out-of-fold) -> residuals
    y_hat = cross_val_predict(LinearRegression(), W, Y, cv=kf)
    # stage 2: predict T from W (out-of-fold) -> residuals
    t_hat = cross_val_predict(XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.1), W, T, cv=kf)
    y_res, t_res = Y - y_hat, T - t_hat

    # stage 3: residual-on-residual linear regression
    final = LinearRegression().fit(t_res.reshape(-1, 1), y_res)
    theta = final.coef_[0]

    # standard error of the slope
    e = y_res - final.predict(t_res.reshape(-1, 1))
    se = np.sqrt(np.sum(t_res**2 * e**2)) / np.sum(t_res**2)

    return theta, se, r2_score(Y, y_hat), r2_score(T, t_hat)

rng = np.random.default_rng(42)
results = {}
for t, w_cols in adj.items():
    T = df[t].values
    W = df[w_cols].values

    theta, se, r2_y, r2_t = dml(Y, T, W)
    placebo_theta, _, _, _ = dml(Y, rng.permutation(T), W)  # shuffled T should give ~0

    results[t] = {
        "ATE": theta,
        "CI_low": theta - 1.96 * se,
        "CI_high": theta + 1.96 * se,
        "R2_y": r2_y,
        "R2_t": r2_t,
        "placebo_ATE": placebo_theta,
    }

print(pd.DataFrame(results).T.round(3))

                          ATE  CI_low  CI_high   R2_y   R2_t  placebo_ATE
study_load              0.007  -0.038    0.051  0.729  0.370       -0.006
living_conditions       0.017  -0.035    0.068  0.729  0.322        0.001
noise_level             0.022  -0.023    0.067  0.730  0.416       -0.013
academic_performance   -0.042  -0.090    0.007  0.738  0.559        0.011
future_career_concerns -0.027  -0.087    0.033  0.738  0.610       -0.012


# Finding 3: every ATE is close to 0. 
## Last assignment we said study load, living conditions, noise level, academic performance, and future career concerns affect stress. DML is suggesting otherwise - tuning any of these 5 doesn't move the needle much.
This is because R^2_y is 73% across the board - counfounders are doing a lot of work to explain stress, not much any of these 5 individually do if 73% of stress is already determined by the confounders.

# Test: try XGBoost & XGBoost, XGBoost & Linear, Linear & XGBoost, Linear & Linear

In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

Y = df["stress_level"].values

# confounders from the data check (likely stress symptoms excluded)
confounders = ["mental_health_history", "safety", "basic_needs", "social_support",
               "teacher_student_relationship", "peer_pressure", "bullying",
               "extracurricular_activities"]

# DAG4 structure among treatments, plus the new confounders
adj = {
    "study_load": confounders,
    "living_conditions": confounders,
    "noise_level": confounders + ["living_conditions"],
    "academic_performance": confounders + ["living_conditions", "study_load"],
    "future_career_concerns": confounders + ["academic_performance"],
}

# our two ML models
models = {
    "Linear": lambda: LinearRegression(),
    "XGB": lambda: XGBRegressor(n_estimators=200, max_depth=3, learning_rate=0.1),
}

# (model_y, model_t) pairs -- A is the original run
specs = {
    "A_Linear_XGB": ("Linear", "XGB"),
    "B_XGB_XGB": ("XGB", "XGB"),
    "C_Linear_Linear": ("Linear", "Linear"),
    "D_XGB_Linear": ("XGB", "Linear"),
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def dml(Y, T, W, model_y, model_t):
    # stage 1: predict Y from W (out-of-fold) -> residuals
    y_hat = cross_val_predict(models[model_y](), W, Y, cv=kf)
    # stage 2: predict T from W (out-of-fold) -> residuals
    t_hat = cross_val_predict(models[model_t](), W, T, cv=kf)
    y_res, t_res = Y - y_hat, T - t_hat

    # stage 3: residual-on-residual linear regression
    final = LinearRegression().fit(t_res.reshape(-1, 1), y_res)
    theta = final.coef_[0]

    # standard error of the slope
    e = y_res - final.predict(t_res.reshape(-1, 1))
    se = np.sqrt(np.sum(t_res**2 * e**2)) / np.sum(t_res**2)

    return theta, se, r2_score(Y, y_hat), r2_score(T, t_hat)

rows = []
for spec, (my, mt) in specs.items():
    rng = np.random.default_rng(42)  # same placebo shuffle for every spec
    for t, w_cols in adj.items():
        T = df[t].values
        W = df[w_cols].values

        theta, se, r2_y, r2_t = dml(Y, T, W, my, mt)
        placebo_theta, _, _, _ = dml(Y, rng.permutation(T), W, my, mt)  # shuffled T should give ~0

        rows.append({
            "spec": spec,
            "treatment": t,
            "ATE": theta,
            "CI_low": theta - 1.96 * se,
            "CI_high": theta + 1.96 * se,
            "R2_y": r2_y,
            "R2_t": r2_t,
            "placebo_ATE": placebo_theta,
        })

results = pd.DataFrame(rows).set_index(["treatment", "spec"]).sort_index()
print(results.round(3))

# side-by-side ATEs across specs
print(results["ATE"].unstack("spec").round(3))

results.to_csv("../results/dml_results.csv")  # saves everything

                                          ATE  CI_low  CI_high   R2_y   R2_t  \
treatment              spec                                                    
academic_performance   A_Linear_XGB    -0.042  -0.090    0.007  0.738  0.559   
                       B_XGB_XGB       -0.063  -0.114   -0.013  0.731  0.559   
                       C_Linear_Linear -0.089  -0.138   -0.039  0.738  0.584   
                       D_XGB_Linear    -0.054  -0.105   -0.004  0.731  0.584   
future_career_concerns A_Linear_XGB    -0.027  -0.087    0.033  0.738  0.610   
                       B_XGB_XGB       -0.036  -0.092    0.020  0.723  0.610   
                       C_Linear_Linear  0.056   0.000    0.111  0.738  0.659   
                       D_XGB_Linear    -0.033  -0.090    0.025  0.723  0.659   
living_conditions      A_Linear_XGB     0.017  -0.035    0.068  0.729  0.322   
                       B_XGB_XGB       -0.006  -0.059    0.047  0.737  0.322   
                       C_Linear_Linear -

## XGB for Y and Linear for T wins!!!!!!

## Best fit, highest T R², catchs non-linear confounding.

## Once confounders are handled properly, most of the causes we used in A3 have no effect on stress. Academic performance has a small effect (really small). Effects we found in A3 are mostly due to confounders we didn't control for (damn). 